# 📊 Notebook 1 — Data Exploration & Understanding
**Project:** Customer Churn Prediction | IBM Telco Dataset  
**Author:** Aditya Bobade  
**Dataset:** 7,043 customers | 21 features | Target: Churn (Yes/No)

---
### Objectives
- Load and inspect the dataset
- Understand feature types and distributions
- Identify missing values and data quality issues
- Explore churn rate and key patterns


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/telco_churn.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST-00001,Male,0,Yes,No,63,Yes,Yes,Fiber optic,Yes,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),83.98,5298.36,No
1,CUST-00002,Female,1,Yes,Yes,3,No,No phone service,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Credit card (automatic),25.36,86.99,No
2,CUST-00003,Male,0,No,Yes,42,Yes,No,DSL,No,...,No,No,No,No,Two year,Yes,Credit card (automatic),56.28,2364.64,No
3,CUST-00004,Male,0,Yes,No,22,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.62,445.83,No
4,CUST-00005,Male,0,Yes,No,37,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Bank transfer (automatic),18.25,676.66,No


In [2]:
# Data types and null check
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
# Statistical summary
df.describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.15,0.36,0.00,0.00,0.00,0.00,1.00
tenure,7043.0,31.84,22.23,1.00,10.00,31.00,52.00,71.00
MonthlyCharges,7043.0,65.85,25.81,18.25,50.52,70.80,86.32,118.75
TotalCharges,7043.0,2096.29,1762.96,18.80,569.94,1561.89,3426.95,7579.50


In [4]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#2ecc71','#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn'); axes[0].set_ylabel('Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v+50, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Rate Breakdown', fontsize=14, fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../charts/01_churn_distribution.png', bbox_inches='tight')
plt.show()
print(f"Churn Rate: {churn_pct['Yes']:.2f}%")


Churn Rate: 28.85%


In [5]:
# Numerical distributions by churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, num_cols):
    ax.hist(df[df['Churn']=='No'][col],  bins=30, alpha=0.6, color='#2ecc71', label='Not Churned')
    ax.hist(df[df['Churn']=='Yes'][col], bins=30, alpha=0.6, color='#e74c3c', label='Churned')
    ax.set_title(f'{col} by Churn', fontweight='bold')
    ax.set_xlabel(col); ax.legend()

plt.suptitle('Numerical Features by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/02_numerical_distributions.png', bbox_inches='tight')
plt.show()


In [6]:
# Churn rate by Contract type
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(contract_churn.index, contract_churn.values,
              color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
for bar, val in zip(bars, contract_churn.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/03_churn_by_contract.png', bbox_inches='tight')
plt.show()


In [7]:
# Churn rate by Internet Service & Payment Method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
    ['InternetService','PaymentMethod'],
    ['Churn by Internet Service','Churn by Payment Method']):
    rates = df.groupby(col)['Churn'].apply(
        lambda x: (x=='Yes').mean()*100).sort_values()
    bars = ax.barh(rates.index, rates.values, color='#3498db', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    for bar, val in zip(bars, rates.values):
        ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../charts/04_churn_by_service_payment.png', bbox_inches='tight')
plt.show()


In [8]:
# Tenure vs MonthlyCharges scatter
fig, ax = plt.subplots(figsize=(10, 6))
for label, color in [('No','#2ecc71'),('Yes','#e74c3c')]:
    g = df[df['Churn']==label]
    ax.scatter(g['tenure'], g['MonthlyCharges'], c=color, alpha=0.35, s=15, label=f'Churn: {label}')
ax.set_title('Tenure vs Monthly Charges by Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (months)'); ax.set_ylabel('Monthly Charges ($)')
ax.legend()
plt.tight_layout()
plt.savefig('../charts/05_tenure_vs_charges_scatter.png', bbox_inches='tight')
plt.show()

print("=== Key EDA Findings ===")
print(f"Churn Rate: {(df['Churn']=='Yes').mean()*100:.2f}%")
print(f"Avg Tenure  - Churned: {df[df['Churn']=='Yes']['tenure'].mean():.1f} mo")
print(f"Avg Tenure  - Retained: {df[df['Churn']=='No']['tenure'].mean():.1f} mo")
print(f"Avg Charges - Churned: ${df[df['Churn']=='Yes']['MonthlyCharges'].mean():.2f}")
print(f"Avg Charges - Retained: ${df[df['Churn']=='No']['MonthlyCharges'].mean():.2f}")


=== Key EDA Findings ===
Churn Rate: 28.85%
Avg Tenure  - Churned: 25.4 mo
Avg Tenure  - Retained: 34.5 mo
Avg Charges - Churned: $72.54
Avg Charges - Retained: $63.13
